# ML Practice Exam — ANSWER KEY
**Note:** Outputs depend on the exact random seed and data. 
Your answers should match to 4 decimal places.
All answers are self-contained — run from top to bottom.


## ⚙️ SETUP — Run This First!

In [6]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Reproducible synthetic "Patient Health" dataset ──────────────
np.random.seed(42)
N = 1200

age           = np.random.randint(25, 75, N).astype(float)
bmi           = np.round(np.random.normal(27, 5, N), 2)
blood_pressure= np.round(np.random.normal(120, 15, N), 1)
cholesterol   = np.round(np.random.normal(200, 40, N), 1)
glucose       = np.round(np.random.normal(100, 25, N), 1)
heart_rate    = np.round(np.random.normal(75, 12, N), 1)
creatinine    = np.round(np.random.normal(1.0, 0.3, N), 3)
hemoglobin    = np.round(np.random.normal(13.5, 1.5, N), 2)

gender       = np.random.choice(['Male','Female'], N)
smoking      = np.random.choice(['Never','Former','Current'], N, p=[0.5,0.3,0.2])
region       = np.random.choice(['North','South','East','West'], N)
risk_level   = np.random.choice(['Low','Medium','High'], N, p=[0.35,0.40,0.25])

# Target: binary classification
log_odds = (-3 + 0.03*age + 0.05*bmi + 0.01*blood_pressure
            + 0.005*cholesterol + 0.01*glucose
            + (smoking=='Current')*0.8 + (risk_level=='High')*1.2)
prob = 1/(1+np.exp(-log_odds))
disease = (np.random.rand(N) < prob).astype(int)

df = pd.DataFrame({
    'age': age, 'bmi': bmi, 'blood_pressure': blood_pressure,
    'cholesterol': cholesterol, 'glucose': glucose,
    'heart_rate': heart_rate, 'creatinine': creatinine,
    'hemoglobin': hemoglobin,
    'gender': gender, 'smoking': smoking,
    'region': region, 'risk_level': risk_level,
    'disease': disease
})

# Inject ~10% missing values in 3 columns
for col, frac in [('bmi',0.08), ('cholesterol',0.10), ('region',0.09)]:
    mask = np.random.choice([True,False], N, p=[frac, 1-frac])
    df.loc[mask, col] = np.nan

print("Dataset shape:", df.shape)
print(df.head())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nClass distribution (disease):")
print(df['disease'].value_counts())


Dataset shape: (1200, 13)
    age    bmi  blood_pressure  cholesterol  glucose  heart_rate  creatinine  \
0  63.0  27.90           132.5        194.6     90.1        82.0       1.454   
1  53.0  40.22           117.3        171.7     71.7        66.9       1.123   
2  39.0  25.99           131.6        210.7     78.7        85.5       1.070   
3  67.0    NaN           143.0          NaN     52.2        65.8       0.659   
4  32.0  35.91           133.8        207.2     97.9        67.4       0.753   

   hemoglobin  gender  smoking region risk_level  disease  
0       14.90  Female   Former   West       High        1  
1       14.26  Female  Current   East        Low        1  
2       11.98  Female    Never   West       High        1  
3       12.95  Female    Never   East     Medium        1  
4       14.73  Female   Former  South     Medium        1  

Missing values per column:
age                 0
bmi                92
blood_pressure      0
cholesterol       102
glucose          

### Q1 [Easy] — Dataset Exploration
After running the setup cell above:
1. How many rows and columns does the dataset have?
2. List all columns that contain missing values and their respective missing counts.
3. What proportion (2 decimal places) of patients have disease = 1?

In [7]:
# Shape
print("Shape:", df.shape)

# Missing per column
missing = df.isnull().sum()
print("\nColumns with missing values:")
print(missing[missing > 0])

# Proportion of disease=1
prop = df['disease'].value_counts(normalize=True)[1]
print(f"\nProportion of disease=1: {prop:.2f}")


Shape: (1200, 13)

Columns with missing values:
bmi             92
cholesterol    102
region         109
dtype: int64

Proportion of disease=1: 0.96


### Q2 [Easy] — Train/Test Split
Split the dataset into features (X) and target (y) where `disease` goes to y and the rest to X.
Then split into train/test with **80:20** ratio using **random_state=42**.

1. How many rows are in X_train?
2. How many positive cases (disease=1) are in y_test?

In [8]:
from sklearn.model_selection import train_test_split

X = df.drop('disease', axis=1)
y = df['disease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("X_train rows:", X_train.shape[0])
print("Positive cases in y_test:", y_test.sum())


X_train rows: 960
Positive cases in y_test: 233


### Q3 [Easy] — Simple Imputation
Use **SimpleImputer** to fill missing values:
- `bmi` → fill with **median**
- `cholesterol` → fill with **mean**
- `region` → fill with **most_frequent**

Apply **fit** only on the train set, then **transform** both train and test.

1. What is the median value (rounded to 2 decimal places) used to fill `bmi`?
2. What is the mean value (rounded to 2 decimal places) used to fill `cholesterol`?
3. What is the most frequent value used to fill `region`?

In [9]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

num_imp = ColumnTransformer([
    ('bmi_imp',  SimpleImputer(strategy='median'),    ['bmi']),
    ('chol_imp', SimpleImputer(strategy='mean'),       ['cholesterol']),
    ('reg_imp',  SimpleImputer(strategy='most_frequent'), ['region']),
], remainder='passthrough')

num_imp.fit(X_train)

bmi_median  = num_imp.named_transformers_['bmi_imp'].statistics_[0]
chol_mean   = num_imp.named_transformers_['chol_imp'].statistics_[0]
reg_mode    = num_imp.named_transformers_['reg_imp'].statistics_[0]

print(f"BMI median used: {bmi_median:.2f}")
print(f"Cholesterol mean used: {chol_mean:.2f}")
print(f"Region most frequent: {reg_mode}")

X_train_imp = num_imp.transform(X_train)
X_test_imp  = num_imp.transform(X_test)


BMI median used: 26.85
Cholesterol mean used: 201.59
Region most frequent: South


### Q4 [Easy] — Feature Scaling
Define `num_cols = ['age','bmi','blood_pressure','cholesterol','glucose','heart_rate','creatinine','hemoglobin']`.

After imputing missing values (Q3 style — use median for bmi, mean for cholesterol), apply the following on **numerical columns only** using the **train** set to fit and **test** set to transform:

- **MinMaxScaler** → store as `X_train_mm`, `X_test_mm`
- **StandardScaler** → store as `X_train_ss`, `X_test_ss`

Answer:
1. What is the **minimum** value in the MinMax-scaled `age` column of X_train? (should be 0.0)
2. What is the **mean** of StandardScaler-scaled `glucose` in X_train? (should be ~0.0 — give to 4 decimal places)
3. What is the **max** absolute value in StandardScaler-scaled `glucose` of X_test?

In [10]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
import pandas as pd

num_cols = ['age','bmi','blood_pressure','cholesterol','glucose',
            'heart_rate','creatinine','hemoglobin']

# First impute, then scale (on num cols only)
imp = SimpleImputer(strategy='median')
X_train_num = imp.fit_transform(X_train[num_cols])
X_test_num  = imp.transform(X_test[num_cols])

X_train_num = pd.DataFrame(X_train_num, columns=num_cols)
X_test_num  = pd.DataFrame(X_test_num,  columns=num_cols)

mm  = MinMaxScaler()
X_train_mm = pd.DataFrame(mm.fit_transform(X_train_num), columns=num_cols)
X_test_mm  = pd.DataFrame(mm.transform(X_test_num),      columns=num_cols)

ss  = StandardScaler()
X_train_ss = pd.DataFrame(ss.fit_transform(X_train_num), columns=num_cols)
X_test_ss  = pd.DataFrame(ss.transform(X_test_num),      columns=num_cols)

print("MinMax age min in X_train:", X_train_mm['age'].min())
print("StandardScaler glucose mean in X_train:", round(X_train_ss['glucose'].mean(), 4))
print("Max abs StandardScaler glucose in X_test:", round(X_test_ss['glucose'].abs().max(), 4))


MinMax age min in X_train: 0.0
StandardScaler glucose mean in X_train: -0.0
Max abs StandardScaler glucose in X_test: 3.6096


### Q5 [Easy-Medium] — Encoding Categorical Variables
Use the following encoders on the **training set** (fit only on train, transform both):

- **OrdinalEncoder** on `risk_level` with explicit category order: `['Low', 'Medium', 'High']`
- **OneHotEncoder** (drop='first') on `gender` and `smoking`
- **pd.get_dummies** (drop_first=True) on `region` — apply to full df for reference only

Answer:
1. What integer does OrdinalEncoder assign to 'High'?
2. How many new columns does OneHotEncoder create for `gender` and `smoking` combined (with drop='first')?

In [11]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# OrdinalEncoder on risk_level
oe = OrdinalEncoder(categories=[['Low','Medium','High']])
train_risk = oe.fit_transform(X_train[['risk_level']])
print("OrdinalEncoder mapping (Low,Medium,High):", oe.categories_)
print("'High' encodes to:", oe.transform([['High']])[0][0])

# OneHotEncoder on gender + smoking
ohe = OneHotEncoder(drop='first', sparse_output=False)
train_ohe = ohe.fit_transform(X_train[['gender','smoking']])
print("\nOneHotEncoder output columns:", ohe.get_feature_names_out(['gender','smoking']))
print("Number of OHE columns:", train_ohe.shape[1])


OrdinalEncoder mapping (Low,Medium,High): [array(['Low', 'Medium', 'High'], dtype=object)]
'High' encodes to: 2.0

OneHotEncoder output columns: ['gender_Male' 'smoking_Former' 'smoking_Never']
Number of OHE columns: 3


### Q6 [Medium] — ColumnTransformer Full Preprocessing Pipeline
Build a **ColumnTransformer** that:
- `num_cols = ['age','bmi','blood_pressure','cholesterol','glucose','heart_rate','creatinine','hemoglobin']`:
  → Impute with **median** then scale with **StandardScaler** (use a Pipeline per step)
- `cat_ohe_cols = ['gender','smoking','region']`:
  → Impute with **most_frequent** then apply **OneHotEncoder(handle_unknown='ignore')**
- `cat_ord_col = ['risk_level']`:
  → **OrdinalEncoder** with order `['Low','Medium','High']`

Use `remainder='drop'` and `verbose_feature_names_out=False`.

Fit on X_train, transform both. Answer:
1. How many total features does the transformed X_train have?
2. What is the transformed value of `age` for the first row of X_train (StandardScaler)? Round to 4 decimal places.

In [12]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_cols     = ['age','bmi','blood_pressure','cholesterol','glucose',
                'heart_rate','creatinine','hemoglobin']
cat_ohe_cols = ['gender','smoking','region']
cat_ord_col  = ['risk_level']

num_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('ss',  StandardScaler())
])

cat_ohe_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

ct = ColumnTransformer([
    ('num',     num_pipe,        num_cols),
    ('cat_ohe', cat_ohe_pipe,    cat_ohe_cols),
    ('cat_ord', OrdinalEncoder(categories=[['Low','Medium','High']]), cat_ord_col),
], remainder='drop', verbose_feature_names_out=False)

X_train_pp = ct.fit_transform(X_train)
X_test_pp  = ct.transform(X_test)

print("Total features after preprocessing:", X_train_pp.shape[1])
print("First row age value (index 0):", round(X_train_pp[0, 0], 4))


Total features after preprocessing: 18
First row age value (index 0): -0.9666


### Q7 [Medium] — KBinsDiscretizer & PolynomialFeatures
**Part A — KBinsDiscretizer:**
Apply `KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')` to the `age` column of X_train (after median imputation).
1. What are the bin edges (rounded to 2 decimal places)?

**Part B — PolynomialFeatures:**
Apply `PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)` to a (n×3) array formed by the first 3 numerical columns (`age`, `bmi`, `blood_pressure`) after StandardScaler.
2. How many output features are produced?

In [13]:
from sklearn.preprocessing import KBinsDiscretizer, PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np

# ── Part A
imp = SimpleImputer(strategy='median')
age_train = imp.fit_transform(X_train[['age']])

kbd = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
kbd.fit(age_train)
print("KBins bin edges (age):", [round(e, 2) for e in kbd.bin_edges_[0]])

# ── Part B
imp3 = SimpleImputer(strategy='median')
three_cols = imp3.fit_transform(X_train[['age','bmi','blood_pressure']])
ss = StandardScaler()
three_scaled = ss.fit_transform(three_cols)

pf = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
out = pf.fit_transform(three_scaled)
print(f"\nPolynomialFeatures(degree=2) on 3 features → {out.shape[1]} output features")
print("Feature names:", pf.get_feature_names_out(['age','bmi','bp']))


KBins bin edges (age): [np.float64(25.0), np.float64(37.75), np.float64(50.0), np.float64(61.0), np.float64(74.0)]

PolynomialFeatures(degree=2) on 3 features → 9 output features
Feature names: ['age' 'bmi' 'bp' 'age^2' 'age bmi' 'age bp' 'bmi^2' 'bmi bp' 'bp^2']


### Q8 [Medium] — Feature Selection
Using the preprocessed data from Q6 (`X_train_pp`, `X_test_pp`, `y_train`, `y_test`):

**Part A — VarianceThreshold:**
Apply `VarianceThreshold(threshold=0.1)`. How many features are **removed**?

**Part B — SelectKBest:**
Apply `SelectKBest(f_classif, k=6)` on `X_train_pp`.
1. What are the indices of the 6 selected features?
2. What is the highest F-statistic score (rounded to 2 decimal places)?

**Part C — SelectPercentile:**
Apply `SelectPercentile(f_classif, percentile=40)`.
How many features are selected?

In [14]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, SelectPercentile, f_classif

# Part A
vt = VarianceThreshold(threshold=0.1)
vt.fit(X_train_pp)
removed = X_train_pp.shape[1] - vt.transform(X_train_pp).shape[1]
print(f"Features removed by VarianceThreshold: {removed}")

# Part B
skb = SelectKBest(f_classif, k=6)
skb.fit(X_train_pp, y_train)
selected_idx = skb.get_support(indices=True)
print(f"\nSelectKBest selected indices: {selected_idx}")
print(f"Highest F-statistic: {skb.scores_[selected_idx].max():.2f}")

# Part C
sp = SelectPercentile(f_classif, percentile=40)
sp.fit(X_train_pp, y_train)
n_selected = sp.transform(X_train_pp).shape[1]
print(f"\nSelectPercentile(40%) features selected: {n_selected}")


Features removed by VarianceThreshold: 0

SelectKBest selected indices: [ 0  3  4 11 12 17]
Highest F-statistic: 12.55

SelectPercentile(40%) features selected: 7


### Q9 [Medium] — RFE & Pipeline Feature Selection
**Part A — RFE:**
Use `RFE(estimator=LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=5)` on `X_train_pp` and `y_train`.
1. Which feature **indices** are selected?
2. What are the ranking values of the first 5 features?

**Part B — Pipeline with SelectKBest:**
Build a `Pipeline` with steps:
1. `SelectKBest(f_classif, k=8)`
2. `StandardScaler()`  ← (data is already scaled but this tests pipeline chaining)
3. `LogisticRegression(max_iter=500, random_state=42)`

Fit on `X_train_pp`/`y_train`, report F1-score on test set (round to 4 decimal places).

In [15]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

# Part A
rfe = RFE(estimator=LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=5)
rfe.fit(X_train_pp, y_train)
print("RFE selected indices:", rfe.get_support(indices=True))
print("RFE rankings (first 5 features):", rfe.ranking_[:5])

# Part B
pipe = Pipeline([
    ('skb', SelectKBest(f_classif, k=8)),
    ('ss',  StandardScaler()),
    ('lr',  LogisticRegression(max_iter=500, random_state=42))
])
pipe.fit(X_train_pp, y_train)
f1 = f1_score(y_test, pipe.predict(X_test_pp))
print(f"\nPipeline F1-score on test: {f1:.4f}")


RFE selected indices: [ 0  3  4 11 16]
RFE rankings (first 5 features): [ 1  4 12  1  1]

Pipeline F1-score on test: 0.9852


### Q10 [Medium] — PCA
Apply `PCA(n_components=5, random_state=42)` on `X_train_pp`.

1. What is the sum of explained variance ratios of all 5 components? (round to 4 decimal places)
2. Which component index (0-based) has the highest explained variance?
3. Now apply `PCA(n_components=0.90)` — how many components are needed to explain 90% variance?

In [16]:
from sklearn.decomposition import PCA

# Fixed n_components=5
pca5 = PCA(n_components=5, random_state=42)
pca5.fit(X_train_pp)
evr = pca5.explained_variance_ratio_
print("Explained variance ratios:", [round(v, 4) for v in evr])
print("Sum of 5 components EVR:", round(sum(evr), 4))
print("Component with highest EVR: index", evr.argmax())

# Variance-threshold PCA
pca90 = PCA(n_components=0.90, random_state=42)
pca90.fit(X_train_pp)
print(f"\nComponents needed for 90% variance: {pca90.n_components_}")


Explained variance ratios: [np.float64(0.1073), np.float64(0.1053), np.float64(0.0998), np.float64(0.096), np.float64(0.0949)]
Sum of 5 components EVR: 0.5033
Component with highest EVR: index 0

Components needed for 90% variance: 11


---
## 🔢 REGRESSION SECTION
### Setup for Regression Questions (Q11–Q18)
Run this cell before attempting Q11–Q18. We switch to the California Housing dataset.

In [18]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

housing = fetch_california_housing()
Xr = pd.DataFrame(housing.data, columns=housing.feature_names)
yr = pd.Series(housing.target, name='MedHouseVal')

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.20, random_state=42)

# Scale features
ss_r = StandardScaler()
Xr_train_s = ss_r.fit_transform(Xr_train)
Xr_test_s  = ss_r.transform(Xr_test)

print("Regression dataset shape:", Xr.shape)
print("Feature names:", housing.feature_names)
print("Train shape:", Xr_train_s.shape, "| Test shape:", Xr_test_s.shape)


Regression dataset shape: (20640, 8)
Feature names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Train shape: (16512, 8) | Test shape: (4128, 8)


### Q11 [Easy-Medium] — Linear Regression
Apply `LinearRegression()` on `Xr_train_s` / `yr_train`.

1. What is the **R² score** on the test set? (round to 4 decimal places)
2. What is the **RMSE** (Root Mean Squared Error) on the test set? (round to 4 decimal places)
3. What is the **coefficient** of the feature `MedInc`? (round to 4 decimal places)
4. What is the **intercept**? (round to 4 decimal places)

In [19]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

lr = LinearRegression()
lr.fit(Xr_train_s, yr_train)

yr_pred = lr.predict(Xr_test_s)
r2   = r2_score(yr_test, yr_pred)
rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))

print(f"R² score (test):   {r2:.4f}")
print(f"RMSE (test):       {rmse:.4f}")

# Coefficient for MedInc (index 0)
medinc_coef = lr.coef_[list(housing.feature_names).index('MedInc')]
print(f"Coefficient MedInc: {medinc_coef:.4f}")
print(f"Intercept:          {lr.intercept_:.4f}")


R² score (test):   0.5758
RMSE (test):       0.7456
Coefficient MedInc: 0.8544
Intercept:          2.0719


### Q12 [Medium] — Ridge vs Lasso vs LinearRegression
Train all three models with default parameters (except `random_state=42` for Ridge/Lasso):

| Model | Params |
|---|---|
| `LinearRegression` | default |
| `Ridge` | `alpha=1.0, random_state=42` |
| `Lasso` | `alpha=0.1, random_state=42` |

1. Which model gives the **lowest MSE** on the test set?
2. What is the MSE of Ridge? (round to 4 decimal places)
3. How many coefficients does Lasso **zero out**?

In [20]:
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_squared_error

models = {
    'LinearRegression': LinearRegression(),
    'Ridge(alpha=1)':   Ridge(alpha=1.0, random_state=42),
    'Lasso(alpha=0.1)': Lasso(alpha=0.1, random_state=42),
}

results = {}
for name, m in models.items():
    m.fit(Xr_train_s, yr_train)
    mse = mean_squared_error(yr_test, m.predict(Xr_test_s))
    results[name] = mse
    print(f"{name:30s} → MSE: {mse:.4f}")

print(f"\nLowest MSE model: {min(results, key=results.get)}")

# Lasso zero coefficients
lasso = models['Lasso(alpha=0.1)']
print(f"Lasso zeroed coefficients: {(lasso.coef_ == 0).sum()}")


LinearRegression               → MSE: 0.5559
Ridge(alpha=1)                 → MSE: 0.5559
Lasso(alpha=0.1)               → MSE: 0.6796

Lowest MSE model: Ridge(alpha=1)
Lasso zeroed coefficients: 5


### Q13 [Medium] — SGDRegressor
Train `SGDRegressor` with the following parameters:
- `loss='squared_error'`
- `penalty='l2'`
- `alpha=0.001`
- `learning_rate='invscaling'`
- `eta0=0.01`
- `max_iter=200`
- `random_state=42`
- `warm_start=False`

1. What is the R² score on the test set? (round to 4 decimal places)
2. What is the R² on the training set? (round to 4 decimal places)

In [21]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score

sgd = SGDRegressor(loss='squared_error', penalty='l2', alpha=0.001,
                   learning_rate='invscaling', eta0=0.01,
                   max_iter=200, random_state=42, warm_start=False)
sgd.fit(Xr_train_s, yr_train)

r2_train = r2_score(yr_train, sgd.predict(Xr_train_s))
r2_test  = r2_score(yr_test,  sgd.predict(Xr_test_s))
print(f"SGDRegressor R² Train: {r2_train:.4f}")
print(f"SGDRegressor R² Test:  {r2_test:.4f}")


SGDRegressor R² Train: 0.6061
SGDRegressor R² Test:  0.5799


### Q14 [Medium-Hard] — cross_val_score
Using `LinearRegression()` and `Xr_train_s` / `yr_train`:
- Run `cross_val_score` with `scoring='r2'`, `cv=5`

1. What is the **maximum** R² across 5 folds? (round to 4 decimal places)
2. What is the **mean** R²? (round to 4 decimal places)
3. What is the **standard deviation** of R² scores? (round to 4 decimal places)

In [22]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

lr = LinearRegression()
scores = cross_val_score(lr, Xr_train_s, yr_train, scoring='r2', cv=5)

print("CV R² scores:", [round(s, 4) for s in scores])
print(f"Max R²:  {scores.max():.4f}")
print(f"Mean R²: {scores.mean():.4f}")
print(f"Std R²:  {scores.std():.4f}")


CV R² scores: [np.float64(0.6201), np.float64(0.613), np.float64(0.6134), np.float64(0.6107), np.float64(0.6002)]
Max R²:  0.6201
Mean R²: 0.6115
Std R²:  0.0065


### Q15 [Hard] — Pipeline: PolynomialFeatures + Ridge + GridSearchCV
Build a `Pipeline`:
1. `PolynomialFeatures(interaction_only=True)`  ← step name: `'poly'`
2. `Ridge()`  ← step name: `'ridge'`

Then apply `GridSearchCV` with:
- `param_grid = {'poly__degree': [1, 2], 'ridge__alpha': [0.1, 1.0, 10.0, 100.0]}`
- `scoring = 'neg_mean_squared_error'`
- `cv = 5`
- `n_jobs = -1`

Fit on `Xr_train_s` / `yr_train`.

1. What is the best `degree` and `alpha`?
2. What is the R² of the best estimator on the test set? (round to 4 decimal places)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

pipe = Pipeline([
    ('poly',  PolynomialFeatures(interaction_only=True)),
    ('ridge', Ridge())
])

param_grid = {
    'poly__degree':  [1, 2],
    'ridge__alpha':  [0.1, 1.0, 10.0, 100.0]
}

gs = GridSearchCV(pipe, param_grid=param_grid,
                  scoring='neg_mean_squared_error', cv=5, n_jobs=-1)
gs.fit(Xr_train_s, yr_train)

print("Best params:", gs.best_params_)

r2 = r2_score(yr_test, gs.best_estimator_.predict(Xr_test_s))
print(f"R² on test (best estimator): {r2:.4f}")


### Q16 [Hard] — SelectKBest(f_regression) + Lasso Pipeline
Build a `Pipeline`:
1. `SelectKBest(f_regression, k=5)` ← step name `'skb'`
2. `Lasso(random_state=42)` ← step name `'lasso'`

Fit on `Xr_train_s` / `yr_train`.

1. What is the **maximum p-value** among all 8 features? (round to 4 decimal places)
2. What are the **indices** of the 5 selected features?
3. What is the R² score on the test set? (round to 4 decimal places)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score

pipe = Pipeline([
    ('skb',   SelectKBest(f_regression, k=5)),
    ('lasso', Lasso(random_state=42))
])
pipe.fit(Xr_train_s, yr_train)

skb_fitted = pipe.named_steps['skb']
print("Max p-value:", round(skb_fitted.pvalues_.max(), 4))
print("Selected feature indices:", skb_fitted.get_support(indices=True))

r2 = r2_score(yr_test, pipe.predict(Xr_test_s))
print(f"Pipeline R² (test): {r2:.4f}")


---
## 🎯 CLASSIFICATION SECTION
### Setup for Classification Questions (Q17–Q30)
Run this cell. We return to the **Patient Health** dataset.
Use the ColumnTransformer-preprocessed data from Q6 — re-run it here for a clean state.

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import pandas as pd

# Re-split (same random_state=42 → same split)
X = df.drop('disease', axis=1)
y = df['disease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

num_cols     = ['age','bmi','blood_pressure','cholesterol','glucose',
                'heart_rate','creatinine','hemoglobin']
cat_ohe_cols = ['gender','smoking','region']
cat_ord_col  = ['risk_level']

num_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('ss',  StandardScaler())
])
cat_ohe_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
ct = ColumnTransformer([
    ('num',     num_pipe,        num_cols),
    ('cat_ohe', cat_ohe_pipe,    cat_ohe_cols),
    ('cat_ord', OrdinalEncoder(categories=[['Low','Medium','High']]), cat_ord_col),
], remainder='drop', verbose_feature_names_out=False)

X_train_pp = ct.fit_transform(X_train)
X_test_pp  = ct.transform(X_test)
print("X_train_pp shape:", X_train_pp.shape)
print("X_test_pp shape:", X_test_pp.shape)


X_train_pp shape: (960, 18)
X_test_pp shape: (240, 18)


### Q17 [Easy-Medium] — Logistic Regression (Specific Params)
Train `LogisticRegression` with:
- `solver='lbfgs'`
- `C=0.5`
- `max_iter=300`
- `tol=1e-4`
- `random_state=42`
- `penalty='l2'`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. What is the **intercept** (bias term) of the model? (round to 4 decimal places)
3. What is the **accuracy** on the test set? (round to 4 decimal places)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

lr = LogisticRegression(solver='lbfgs', C=0.5, max_iter=300,
                        tol=1e-4, random_state=42, penalty='l2')
lr.fit(X_train_pp, y_train)

f1  = f1_score(y_test, lr.predict(X_test_pp))
acc = accuracy_score(y_test, lr.predict(X_test_pp))

print(f"F1-score (test):   {f1:.4f}")
print(f"Intercept:         {lr.intercept_[0]:.4f}")
print(f"Accuracy (test):   {acc:.4f}")


F1-score (test):   0.9852
Intercept:         3.3277
Accuracy (test):   0.9708


### Q18 [Medium] — GridSearchCV with Logistic Regression
Use `GridSearchCV` with `LogisticRegression(random_state=42)` and the following grid:
- `solver`: `['lbfgs', 'sag']`
- `C`: `[0.01, 0.1, 1, 10]`
- `max_iter`: `[100, 200, 500]`
- `cv = 4`
- `scoring = 'f1'` (default)

1. What is the **best value of C**?
2. What is the **best solver**?
3. What is the **precision score** on the test set using the best estimator? (round to 4 decimal places)
4. What is the **recall score**? (round to 4 decimal places)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

param_grid = {
    'solver':   ['lbfgs', 'sag'],
    'C':        [0.01, 0.1, 1, 10],
    'max_iter': [100, 200, 500]
}

gs = GridSearchCV(LogisticRegression(random_state=42),
                  param_grid=param_grid, cv=4, scoring='f1')
gs.fit(X_train_pp, y_train)

print("Best params:", gs.best_params_)
best = gs.best_estimator_

prec = precision_score(y_test, best.predict(X_test_pp))
rec  = recall_score(y_test,    best.predict(X_test_pp))
print(f"Precision (test): {prec:.4f}")
print(f"Recall (test):    {rec:.4f}")


### Q19 [Medium] — SGDClassifier
Train `SGDClassifier` with:
- `loss='log_loss'`
- `penalty='l2'`
- `alpha=0.001`
- `max_iter=100`
- `learning_rate='constant'`
- `eta0=0.01`
- `random_state=42`
- `warm_start=False`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. What is the **accuracy** on the test set? (round to 4 decimal places)

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score

sgd = SGDClassifier(loss='log_loss', penalty='l2', alpha=0.001,
                    max_iter=100, learning_rate='constant', eta0=0.01,
                    random_state=42, warm_start=False)
sgd.fit(X_train_pp, y_train)

f1  = f1_score(y_test, sgd.predict(X_test_pp))
acc = accuracy_score(y_test, sgd.predict(X_test_pp))
print(f"SGDClassifier F1-score: {f1:.4f}")
print(f"SGDClassifier Accuracy: {acc:.4f}")


### Q20 [Medium] — KNeighborsClassifier + GridSearchCV
Use `GridSearchCV` with `KNeighborsClassifier()`:
- `n_neighbors`: `[3, 5, 7, 9, 11]`
- `metric`: `['euclidean', 'manhattan']`
- `cv = 5`
- `scoring = 'f1'`

1. What is the **best K** (n_neighbors)?
2. What is the **best metric**?
3. What is the **F1-score** of the best model on the test set? (round to 4 decimal places)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'metric':      ['euclidean', 'manhattan']
}

gs = GridSearchCV(KNeighborsClassifier(), param_grid=param_grid, cv=5, scoring='f1')
gs.fit(X_train_pp, y_train)

print("Best params:", gs.best_params_)
f1 = f1_score(y_test, gs.best_estimator_.predict(X_test_pp))
print(f"KNN Best F1 (test): {f1:.4f}")


### Q21 [Medium] — Naive Bayes
Train `GaussianNB()` (default params) on `X_train_pp` / `y_train`.

1. What is the **accuracy** on the test set? (round to 4 decimal places)
2. What is the **F1-score**? (round to 4 decimal places)
3. What is the **AUC-ROC** score? (round to 4 decimal places)
   Hint: use `predict_proba` to get probabilities.

In [25]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

gnb = GaussianNB()
gnb.fit(X_train_pp, y_train)

y_pred  = gnb.predict(X_test_pp)
y_proba = gnb.predict_proba(X_test_pp)[:, 1]

print(f"GaussianNB Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"GaussianNB F1-score:  {f1_score(y_test, y_pred):.4f}")
print(f"GaussianNB AUC-ROC:   {roc_auc_score(y_test, y_proba):.4f}")


GaussianNB Accuracy:  0.9708
GaussianNB F1-score:  0.9852
GaussianNB AUC-ROC:   0.5880


### Q22 [Medium-Hard] — Support Vector Machine
Train `SVC` with:
- `kernel='rbf'`
- `C=1.0`
- `gamma='scale'`
- `random_state=42`
- `probability=True`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. What is the **AUC-ROC** score? (round to 4 decimal places)
3. Now use `GridSearchCV` with `SVC(kernel='rbf', random_state=42, probability=True)`:
   - `C`: `[0.1, 1, 10]`
   - `gamma`: `['scale', 'auto']`
   - `cv=3`, `scoring='f1'`
   What are the **best C and gamma**?

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, roc_auc_score

svc = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, probability=True)
svc.fit(X_train_pp, y_train)

f1  = f1_score(y_test, svc.predict(X_test_pp))
auc = roc_auc_score(y_test, svc.predict_proba(X_test_pp)[:, 1])
print(f"SVC F1-score:  {f1:.4f}")
print(f"SVC AUC-ROC:   {auc:.4f}")

# GridSearchCV
param_grid = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']}
gs = GridSearchCV(SVC(kernel='rbf', random_state=42, probability=True),
                  param_grid=param_grid, cv=3, scoring='f1')
gs.fit(X_train_pp, y_train)
print("\nBest SVC params:", gs.best_params_)


### Q23 [Medium] — Confusion Matrix Deep Dive
Using the **Logistic Regression from Q17** (refit if needed):
- Compute the **confusion matrix**
- Report: **TP, TN, FP, FN**
- Compute manually: Precision, Recall, F1 (verify against sklearn)
- What is the **False Positive Rate** (FPR = FP / (FP+TN))? (round to 4 decimal places)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
import numpy as np

lr = LogisticRegression(solver='lbfgs', C=0.5, max_iter=300,
                         tol=1e-4, random_state=42, penalty='l2')
lr.fit(X_train_pp, y_train)
y_pred = lr.predict(X_test_pp)

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

print("Confusion Matrix:")
print(cm)
print(f"\nTN={TN}, FP={FP}, FN={FN}, TP={TP}")

precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)
fpr       = FP / (FP + TN)

print(f"\nPrecision (manual): {precision:.4f}")
print(f"Recall (manual):    {recall:.4f}")
print(f"F1 (manual):        {f1:.4f}")
print(f"FPR:                {fpr:.4f}")


---
## 🌳 TREES & ENSEMBLES SECTION (Q24–Q31)
(Continue using X_train_pp, X_test_pp, y_train, y_test from the Classification Setup cell)

In [ ]:
# Nothing to run — data already available from Classification Setup cell
print("Data shapes:")
print("X_train_pp:", X_train_pp.shape)
print("X_test_pp: ", X_test_pp.shape)


### Q24 [Medium] — DecisionTreeClassifier + GridSearchCV
Use `GridSearchCV` with `DecisionTreeClassifier(random_state=42)`:
- `criterion`: `['gini', 'entropy']`
- `max_depth`: `[4, 5, 6, 7, 8]`
- `min_samples_leaf`: `[5, 10, 15, 20]`
- `min_samples_split`: `[10, 20, 30]`
- `cv = 3`

1. What are the **best hyperparameters**?
2. What is the **depth** of the best estimator?
3. How many **leaves** does the best tree have? (use `.get_n_leaves()`)
4. What is the **accuracy** on the test set? (round to 4 decimal places)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

param_grid = {
    'criterion':        ['gini','entropy'],
    'max_depth':        [4, 5, 6, 7, 8],
    'min_samples_leaf': [5, 10, 15, 20],
    'min_samples_split':[10, 20, 30]
}

gs = GridSearchCV(DecisionTreeClassifier(random_state=42),
                  param_grid=param_grid, cv=3)
gs.fit(X_train_pp, y_train)

best_dt = gs.best_estimator_
print("Best params:", gs.best_params_)
print(f"Tree depth:  {best_dt.get_depth()}")
print(f"N leaves:    {best_dt.get_n_leaves()}")

acc = accuracy_score(y_test, best_dt.predict(X_test_pp))
print(f"Accuracy (test): {acc:.4f}")


### Q25 [Medium-Hard] — Decision Tree: Two Approaches Comparison
Train two `DecisionTreeClassifier` models (both with `random_state=42`, `criterion='entropy'`):

- **Approach A**: `max_depth=8, min_samples_split=40`
- **Approach B**: `ccp_alpha=0.005` (cost complexity pruning)

1. Which approach gives **higher accuracy** on the test set?
2. What is the **absolute difference in True Positives (TP)** between both approaches?
3. What is the **sum of total nodes** (node_count) of both trees?

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np

appA = DecisionTreeClassifier(criterion='entropy', max_depth=8,
                               min_samples_split=40, random_state=42)
appA.fit(X_train_pp, y_train)
accA = accuracy_score(y_test, appA.predict(X_test_pp))

appB = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.005, random_state=42)
appB.fit(X_train_pp, y_train)
accB = accuracy_score(y_test, appB.predict(X_test_pp))

print(f"Approach A accuracy: {accA:.4f}")
print(f"Approach B accuracy: {accB:.4f}")
print(f"Better approach: {'A' if accA >= accB else 'B'}")

cmA = confusion_matrix(y_test, appA.predict(X_test_pp))
cmB = confusion_matrix(y_test, appB.predict(X_test_pp))
tp_diff = abs(int(cmA[1,1]) - int(cmB[1,1]))
print(f"\n|TP_A - TP_B|: {tp_diff}")

node_sum = appA.tree_.node_count + appB.tree_.node_count
print(f"Sum of node counts: {node_sum}")


### Q26 [Medium-Hard] — RandomForestClassifier
Train `RandomForestClassifier` with:
- `n_estimators=100`
- `max_depth=8`
- `min_samples_leaf=5`
- `oob_score=True`
- `random_state=42`
- `n_jobs=-1`

1. What is the **OOB score**? (round to 4 decimal places)
2. What is the **F1-score** on the test set? (round to 4 decimal places)
3. Which feature has the **highest importance**? Print feature index and importance value.
4. What is the **AUC-ROC** on the test set? (round to 4 decimal places)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np

rf = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=5,
                             oob_score=True, random_state=42, n_jobs=-1)
rf.fit(X_train_pp, y_train)

y_pred  = rf.predict(X_test_pp)
y_proba = rf.predict_proba(X_test_pp)[:, 1]

print(f"OOB Score:     {rf.oob_score_:.4f}")
print(f"F1-score:      {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC:       {roc_auc_score(y_test, y_proba):.4f}")

fi = rf.feature_importances_
top_idx = fi.argmax()
print(f"\nTop feature index: {top_idx}, importance: {fi[top_idx]:.4f}")


### Q27 [Medium-Hard] — BaggingClassifier + VotingClassifier
**Part A — Bagging:**
Train `BaggingClassifier` with:
- `estimator=DecisionTreeClassifier(random_state=42)`
- `n_estimators=50`
- `max_samples=0.8`
- `max_features=0.8`
- `oob_score=True`
- `random_state=42`

1. OOB score? (round to 4 decimal places)
2. F1-score on test? (round to 4 decimal places)

**Part B — VotingClassifier (Hard & Soft):**
Combine `LogisticRegression(max_iter=500, random_state=42)`, `GaussianNB()`, and `DecisionTreeClassifier(max_depth=5, random_state=42)`.
3. Which voting type (`'hard'` or `'soft'`) gives **higher accuracy** on the test set?

In [ ]:
from sklearn.ensemble import BaggingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score, accuracy_score

# Part A
bag = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                        n_estimators=50, max_samples=0.8, max_features=0.8,
                        oob_score=True, random_state=42)
bag.fit(X_train_pp, y_train)
print(f"Bagging OOB score: {bag.oob_score_:.4f}")
print(f"Bagging F1-score:  {f1_score(y_test, bag.predict(X_test_pp)):.4f}")

# Part B
estimators = [
    ('lr',  LogisticRegression(max_iter=500, random_state=42)),
    ('gnb', GaussianNB()),
    ('dt',  DecisionTreeClassifier(max_depth=5, random_state=42))
]

for voting in ['hard','soft']:
    vc = VotingClassifier(estimators=estimators, voting=voting)
    vc.fit(X_train_pp, y_train)
    acc = accuracy_score(y_test, vc.predict(X_test_pp))
    print(f"VotingClassifier({voting}) accuracy: {acc:.4f}")


### Q28 [Hard] — AdaBoost & GradientBoosting
**Part A — AdaBoostClassifier:**
- `n_estimators=100`
- `learning_rate=0.5`
- `algorithm='SAMME'`
- `random_state=42`
- Base estimator: `DecisionTreeClassifier(max_depth=2, random_state=42)`

1. F1-score on test? (round to 4 decimal places)

**Part B — GradientBoostingClassifier with GridSearchCV:**
- `n_estimators`: `[50, 100]`
- `max_depth`: `[2, 3]`
- `learning_rate`: `[0.05, 0.1]`
- `cv=3`, `scoring='f1'`, `random_state=42`

2. What are the best params?
3. F1-score of best model on test? (round to 4 decimal places)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# Part A
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
    n_estimators=100, learning_rate=0.5, algorithm='SAMME', random_state=42)
ada.fit(X_train_pp, y_train)
print(f"AdaBoost F1-score: {f1_score(y_test, ada.predict(X_test_pp)):.4f}")

# Part B
param_grid = {
    'n_estimators':  [50, 100],
    'max_depth':     [2, 3],
    'learning_rate': [0.05, 0.1]
}
gs = GridSearchCV(GradientBoostingClassifier(random_state=42),
                  param_grid=param_grid, cv=3, scoring='f1')
gs.fit(X_train_pp, y_train)
print("\nGradientBoosting best params:", gs.best_params_)
print(f"GradientBoosting F1 (test):   {f1_score(y_test, gs.best_estimator_.predict(X_test_pp)):.4f}")


### Q29 [Hard] — XGBoost
Train `XGBClassifier` with:
- `n_estimators=100`
- `max_depth=4`
- `learning_rate=0.1`
- `subsample=0.8`
- `colsample_bytree=0.8`
- `use_label_encoder=False`
- `eval_metric='logloss'`
- `random_state=42`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. What is the **AUC-ROC** score? (round to 4 decimal places)
3. What is the index of the **most important feature** according to `feature_importances_`?

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np

xgb = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                     subsample=0.8, colsample_bytree=0.8,
                     eval_metric='logloss', random_state=42,
                     verbosity=0)
xgb.fit(X_train_pp, y_train)

y_pred  = xgb.predict(X_test_pp)
y_proba = xgb.predict_proba(X_test_pp)[:, 1]

print(f"XGBoost F1-score: {f1_score(y_test, y_pred):.4f}")
print(f"XGBoost AUC-ROC:  {roc_auc_score(y_test, y_proba):.4f}")

top_idx = xgb.feature_importances_.argmax()
print(f"Most important feature index: {top_idx}")
print(f"Importance value: {xgb.feature_importances_[top_idx]:.4f}")


---
## 🔵 CLUSTERING SECTION (Q30–Q32)
For clustering, use only the numerical features of df (after imputation & scaling).
Run this setup first.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import numpy as np

num_cols = ['age','bmi','blood_pressure','cholesterol','glucose',
            'heart_rate','creatinine','hemoglobin']

imp_cl = SimpleImputer(strategy='median')
X_cl   = imp_cl.fit_transform(df[num_cols])
ss_cl  = StandardScaler()
X_cl_s = ss_cl.fit_transform(X_cl)
print("Clustering data shape:", X_cl_s.shape)


### Q30 [Medium] — KMeans Clustering
Train `KMeans` with:
- `n_clusters=3`
- `init='k-means++'`
- `n_init=10`
- `max_iter=300`
- `random_state=42`

1. What is the **inertia** (within-cluster sum of squares)? (round to 2 decimal places)
2. How many samples fall in each cluster? (sorted by cluster index)
3. Now run KMeans for `k = 2, 3, 4, 5, 6` and print the inertia for each.
   (This is the elbow method — use this to identify the optimal k visually.)

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

km = KMeans(n_clusters=3, init='k-means++', n_init=10,
            max_iter=300, random_state=42)
km.fit(X_cl_s)

print(f"Inertia (k=3): {km.inertia_:.2f}")
labels, counts = np.unique(km.labels_, return_counts=True)
print("Cluster sizes:", dict(zip(labels, counts)))

# Elbow method
print("\nElbow Method:")
for k in range(2, 7):
    km_k = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_k.fit(X_cl_s)
    print(f"  k={k} → inertia={km_k.inertia_:.2f}")


### Q31 [Medium-Hard] — Hierarchical Agglomerative Clustering
Apply `AgglomerativeClustering` with:
- `n_clusters=3`
- `linkage='ward'`
- `metric='euclidean'`

1. How many samples are in each cluster?
2. Now use `n_clusters=None, distance_threshold=5, linkage='ward'` — how many clusters are formed automatically?
3. Compute the **silhouette score** for both settings. Which gives a higher silhouette score?

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import numpy as np

# Fixed n_clusters=3
hac3 = AgglomerativeClustering(n_clusters=3, linkage='ward', metric='euclidean')
labels3 = hac3.fit_predict(X_cl_s)
vals, cnts = np.unique(labels3, return_counts=True)
print("HAC(n=3) cluster sizes:", dict(zip(vals, cnts)))
sil3 = silhouette_score(X_cl_s, labels3)
print(f"Silhouette score (n=3): {sil3:.4f}")

# Auto threshold
hac_auto = AgglomerativeClustering(n_clusters=None, distance_threshold=5,
                                    linkage='ward', metric='euclidean')
labels_auto = hac_auto.fit_predict(X_cl_s)
n_cl_auto = len(np.unique(labels_auto))
print(f"\nHAC(threshold=5) → {n_cl_auto} clusters")
sil_auto = silhouette_score(X_cl_s, labels_auto)
print(f"Silhouette score (threshold): {sil_auto:.4f}")
print(f"Better silhouette: {'n=3' if sil3 >= sil_auto else 'threshold'}")


---
## 🧠 MLP SECTION (Q32–Q34)
(Use X_train_pp, X_test_pp, y_train, y_test from Classification Setup)

In [ ]:
print("Using classification preprocessed data.")
print("X_train_pp shape:", X_train_pp.shape)


### Q32 [Hard] — MLPClassifier (Specific Params)
Train `MLPClassifier` with:
- `hidden_layer_sizes=(64, 32)`
- `activation='relu'`
- `solver='adam'`
- `alpha=0.001`
- `batch_size=64`
- `learning_rate_init=0.001`
- `max_iter=200`
- `random_state=42`
- `early_stopping=True`
- `validation_fraction=0.1`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. How many **iterations** did training run for? (`.n_iter_`)
3. What is the **best validation score** achieved during training? (`.best_validation_score_`, round to 4 decimal places)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                    solver='adam', alpha=0.001, batch_size=64,
                    learning_rate_init=0.001, max_iter=200,
                    random_state=42, early_stopping=True,
                    validation_fraction=0.1)
mlp.fit(X_train_pp, y_train)

print(f"MLP F1-score:             {f1_score(y_test, mlp.predict(X_test_pp)):.4f}")
print(f"Iterations run:           {mlp.n_iter_}")
print(f"Best validation score:    {mlp.best_validation_score_:.4f}")


### Q33 [Hard] — MLPClassifier + GridSearchCV
Use `GridSearchCV` with `MLPClassifier(random_state=42, max_iter=200)`:
- `hidden_layer_sizes`: `[(32,), (64,), (64, 32), (128, 64)]`
- `alpha`: `[0.0001, 0.001, 0.01]`
- `solver`: `['adam', 'sgd']`
- `cv = 3`, `scoring = 'f1'`

1. What are the **best hyperparameters**?
2. What is the **F1-score** of the best model on the test set? (round to 4 decimal places)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (64, 32), (128, 64)],
    'alpha':              [0.0001, 0.001, 0.01],
    'solver':             ['adam', 'sgd']
}

gs = GridSearchCV(MLPClassifier(random_state=42, max_iter=200),
                  param_grid=param_grid, cv=3, scoring='f1')
gs.fit(X_train_pp, y_train)

print("Best MLP params:", gs.best_params_)
print(f"Best MLP F1 (test): {f1_score(y_test, gs.best_estimator_.predict(X_test_pp)):.4f}")


### Q34 [Hard] — MLPRegressor on California Housing
Using the regression data (`Xr_train_s`, `Xr_test_s`, `yr_train`, `yr_test`):

Train `MLPRegressor` with:
- `hidden_layer_sizes=(100, 50)`
- `activation='relu'`
- `solver='adam'`
- `alpha=0.001`
- `max_iter=300`
- `random_state=42`
- `early_stopping=True`

1. What is the **R² score** on the test set? (round to 4 decimal places)
2. What is the **RMSE** on the test set? (round to 4 decimal places)

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

mlpr = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu',
                    solver='adam', alpha=0.001, max_iter=300,
                    random_state=42, early_stopping=True)
mlpr.fit(Xr_train_s, yr_train)

yr_pred = mlpr.predict(Xr_test_s)
r2   = r2_score(yr_test, yr_pred)
rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))

print(f"MLPRegressor R² (test):   {r2:.4f}")
print(f"MLPRegressor RMSE (test): {rmse:.4f}")


---
## ⚖️ HANDLING IMBALANCE (Q35)
### Q35 [Hard] — SMOTE + RandomForest Pipeline
Create a synthetic **imbalanced** version of our dataset (minority class = 15%).
Then compare performance **before and after SMOTE**.

Setup:
```python
from imblearn.over_sampling import SMOTE
# Use X_train_pp, y_train but artificially make it imbalanced for demonstration
```

Steps:
1. Apply `SMOTE(random_state=42)` to `X_train_pp` / `y_train`.
2. Print class distribution **before** and **after** SMOTE.
3. Train `RandomForestClassifier(n_estimators=100, random_state=42)` on:
   - Original `X_train_pp` / `y_train`
   - SMOTE-resampled data
4. Report **F1-score** for both. Which is better?

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import numpy as np

# Create imbalanced training data (keep only 20% of class 1)
np.random.seed(42)
idx_0 = np.where(y_train == 0)[0]
idx_1 = np.where(y_train == 1)[0]
idx_1_reduced = np.random.choice(idx_1, size=int(len(idx_1)*0.3), replace=False)
idx_imb = np.concatenate([idx_0, idx_1_reduced])

X_imb = X_train_pp[idx_imb]
y_imb = np.array(y_train)[idx_imb]

print("Before SMOTE:")
vals, cnts = np.unique(y_imb, return_counts=True)
print(dict(zip(vals, cnts)))

# SMOTE
sm = SMOTE(random_state=42)
X_smote, y_smote = sm.fit_resample(X_imb, y_imb)
print("\nAfter SMOTE:")
vals2, cnts2 = np.unique(y_smote, return_counts=True)
print(dict(zip(vals2, cnts2)))

# Train & compare
rf_orig  = RandomForestClassifier(n_estimators=100, random_state=42)
rf_orig.fit(X_imb, y_imb)
f1_orig = f1_score(y_test, rf_orig.predict(X_test_pp))

rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_smote, y_smote)
f1_smote = f1_score(y_test, rf_smote.predict(X_test_pp))

print(f"\nF1 without SMOTE: {f1_orig:.4f}")
print(f"F1 with SMOTE:    {f1_smote:.4f}")
print(f"Better model: {'SMOTE' if f1_smote > f1_orig else 'Original'}")


---
## 🏆 BONUS HARD QUESTION (Q36)
### Q36 [Hard] — End-to-End Pipeline: Preprocessing → Feature Selection → Model → Hyperparameter Tuning
Build a **single end-to-end Pipeline** that:

1. **Preprocessing**: `ColumnTransformer` (as in Q6 — impute + scale numericals, OHE categoricals, Ordinal encode risk_level)
2. **Feature Selection**: `SelectKBest(f_classif)` with `k` as a hyperparameter
3. **Classifier**: `RandomForestClassifier(random_state=42)`

Use `GridSearchCV` with:
- `select__k`: `[8, 10, 12]`
- `clf__n_estimators`: `[50, 100]`
- `clf__max_depth`: `[5, 8, None]`
- `cv=3`, `scoring='f1'`, `n_jobs=-1`

Fit on raw `X_train` / `y_train` (not the preprocessed version — the pipeline handles it).

1. What are the **best hyperparameters**?
2. What is the **F1-score** of the best end-to-end pipeline on `X_test`/`y_test`? (round to 4 decimal places)
3. What is the **AUC-ROC**? (round to 4 decimal places)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, roc_auc_score

num_cols     = ['age','bmi','blood_pressure','cholesterol','glucose',
                'heart_rate','creatinine','hemoglobin']
cat_ohe_cols = ['gender','smoking','region']
cat_ord_col  = ['risk_level']

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('ss',  StandardScaler())])
cat_ohe_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocess = ColumnTransformer([
    ('num',     num_pipe,        num_cols),
    ('cat_ohe', cat_ohe_pipe,    cat_ohe_cols),
    ('cat_ord', OrdinalEncoder(categories=[['Low','Medium','High']]), cat_ord_col),
], remainder='drop', verbose_feature_names_out=False)

full_pipe = Pipeline([
    ('preprocess', preprocess),
    ('select',     SelectKBest(f_classif)),
    ('clf',        RandomForestClassifier(random_state=42))
])

param_grid = {
    'select__k':          [8, 10, 12],
    'clf__n_estimators':  [50, 100],
    'clf__max_depth':     [5, 8, None]
}

gs = GridSearchCV(full_pipe, param_grid=param_grid,
                  cv=3, scoring='f1', n_jobs=-1)
gs.fit(X_train, y_train)

best = gs.best_estimator_
print("Best params:", gs.best_params_)

y_pred  = best.predict(X_test)
y_proba = best.predict_proba(X_test)[:, 1]

print(f"\nEnd-to-End Pipeline F1-score: {f1_score(y_test, y_pred):.4f}")
print(f"End-to-End Pipeline AUC-ROC:  {roc_auc_score(y_test, y_proba):.4f}")
